# 04 Reproduce H→bb baseline results (working notebook)

This notebook reproduces the **same H→bb/STXS χ² logic** used in `stxs_functions.py` and visualized in `fit_plots.ipynb`, but in a robust way that works from your cloned repo path.

## Critical fix vs previous broken version
`sf.stxs_fit(coffea_name, ...)` expects `coffea/<name>.coffea` relative to current working directory and also relies on internal hard-coded paths.

This notebook avoids that fragility by:
1. Loading coffea by **full path** from `asset_manifest.json`.
2. Reusing the same mathematical model from `stxs_functions.py` (quadratic fit + χ² definition).
3. Producing outputs/plots in the same style as `fit_plots.ipynb`.

In [ ]:
from pathlib import Path
import json, sys
import numpy as np, pandas as pd, matplotlib.pyplot as plt, mplhep as hep
from matplotlib.lines import Line2D
from coffea import util
from scipy.optimize import curve_fit
hep.style.use('CMS')

def find_repo_root(start=None, marker='smeft_contents.txt', max_up=8):
    p=Path(start or Path.cwd()).resolve()
    for _ in range(max_up+1):
        if (p/marker).exists(): return p
        if p.parent==p: break
        p=p.parent
    raise FileNotFoundError(f'Could not find {marker} from {Path.cwd()}')

REPO=find_repo_root()
NBDIR=REPO/'notebooks_hww_fit'
manifest=json.loads((NBDIR/'asset_manifest.json').read_text())
coffea_fullpath=manifest['default_coffea_fullpath']
print('repo:',REPO)
print('coffea_fullpath:',coffea_fullpath)

In [ ]:
# Import stxs_functions for helper pieces (wc map and bin-yield extraction).
for pth in ['/uscms_data/d3/azhou/smeft/analysis','/uscms_data/d3/azhou/smeft/analysis/hbb-coffea']:
    if pth not in sys.path: sys.path.append(pth)
import stxs_functions as sf

# Operator list used in legacy fit_plots.ipynb
SMEFT_OPS=['cHbox','cHDD','cHj1','cHj3','cHu','cHd','cHudRe','cuWRe','cuBRe','cdWRe','cdBRe','cHW','cHB','cHWB']
SMEFTCPV_OPS=['cHudIm','cuWIm','cdWIm','cuBIm','cdBIm','cHWtil','cHBtil','cHWBtil']
ALL_OPS=SMEFT_OPS+SMEFTCPV_OPS
print('n operators:',len(ALL_OPS))

In [ ]:
# ---- Exact Hbb chi2 model from stxs_functions.py ----
def chisq_stxs_hbb(calculated_xsec1, calculated_xsec2):
    cms_obs = np.array([240., 120., 200., 190., 68., 61.])
    bin1_obs, bin2_obs = cms_obs[0], cms_obs[1]
    sig1_up, sig1_down, sig2_up, sig2_down = cms_obs[2], cms_obs[3], cms_obs[4], cms_obs[5]

    bin1_diff = calculated_xsec1 - bin1_obs
    bin2_diff = calculated_xsec2 - bin2_obs
    sigma1 = sig1_up if bin1_diff >= 0 else sig1_down
    sigma2 = sig2_up if bin2_diff >= 0 else sig2_down

    bin1_chisq = ((calculated_xsec1 - bin1_obs)/sigma1)**2
    bin2_chisq = ((calculated_xsec2 - bin2_obs)/sigma2)**2
    total_chisq = bin1_chisq + bin2_chisq
    return bin1_chisq, bin2_chisq, total_chisq

def stxs_reweight_coeffs_fullpath(coffea_path, operator_list):
    all_h = util.load(coffea_path)['htxs']
    wc_mapping = sf.wc_map_dict(operator_list)

    bin1_yield, bin2_yield = {}, {}
    for p,wc_label in wc_mapping.items():
        if wc_label not in list(all_h.axes['wc']):
            continue
        hslice = all_h[{'wc': wc_label}]
        b1,b2 = sf.get_bin_yields(hslice)
        bin1_yield[p]=b1
        bin2_yield[p]=b2

    pts=[p for p in wc_mapping.keys() if p in bin1_yield and p in bin2_yield]
    if len(operator_list)==1:
        x=np.array([p[0] for p in pts])
        y1=np.array([bin1_yield[p] for p in pts]); y2=np.array([bin2_yield[p] for p in pts])
        c1,_=curve_fit(sf.quad_1D, x, y1, p0=[1,1,1])
        c2,_=curve_fit(sf.quad_1D, x, y2, p0=[1,1,1])
    elif len(operator_list)==2:
        x1=np.array([p[0] for p in pts]); x2=np.array([p[1] for p in pts])
        y1=np.array([bin1_yield[p] for p in pts]); y2=np.array([bin2_yield[p] for p in pts])
        c1,_=curve_fit(sf.quad_2D, (x1,x2), y1, p0=[1,1,1,1,1,1])
        c2,_=curve_fit(sf.quad_2D, (x1,x2), y2, p0=[1,1,1,1,1,1])
    else:
        raise ValueError('Only 1D/2D supported')
    return c1,c2

In [ ]:
def hbb_stxs_fit_fullpath(coffea_path, operator_list, wc_min=-100, wc_max=100, n=500, mg_sigma_pb=3.594):
    if len(operator_list)>2:
        raise ValueError('Only <=2 operators supported, matching legacy stxs_fit')

    c1,c2 = stxs_reweight_coeffs_fullpath(coffea_path, operator_list)
    sumw = util.load(coffea_path)['sumw_all_noEW'].value

    wc_space_dict={}
    if len(operator_list)==1:
        for x in np.linspace(wc_min,wc_max,n):
            b1 = mg_sigma_pb*1000*sf.quad_1D(x,*c1)/sumw
            b2 = mg_sigma_pb*1000*sf.quad_1D(x,*c2)/sumw
            bt1,bt2,btt = chisq_stxs_hbb(b1,b2)
            wc_space_dict[float(x)]={'bin1':[float(b1),float(bt1)],'bin2':[float(b2),float(bt2)],'total':[float(b1+b2),float(btt)]}
    else:
        g=np.linspace(wc_min,wc_max,n)
        for x in g:
            for y in g:
                b1 = mg_sigma_pb*1000*sf.quad_2D((x,y),*c1)/sumw
                b2 = mg_sigma_pb*1000*sf.quad_2D((x,y),*c2)/sumw
                bt1,bt2,btt = chisq_stxs_hbb(b1,b2)
                wc_space_dict[(float(x),float(y))]={'bin1':[float(b1),float(bt1)],'bin2':[float(b2),float(bt2)],'total':[float(b1+b2),float(btt)]}
    return wc_space_dict

In [ ]:
def hbb_fit_1d(op, coffea_path=coffea_fullpath, wc_min=-100, wc_max=100, n=500):
    ws = hbb_stxs_fit_fullpath(coffea_path, [op], wc_min=wc_min, wc_max=wc_max, n=n)
    x=np.array(sorted(ws.keys()),dtype=float)
    y=np.array([ws[p]['total'][1] for p in x],dtype=float)
    return pd.DataFrame({'op':op,'wc':x,'chi2_hbb':y}), ws

def run_all_hbb_1d(ops=ALL_OPS, **scan_kwargs):
    out_df, out_ws = {}, {}
    for op in ops:
        try:
            d,w = hbb_fit_1d(op, **scan_kwargs)
            out_df[op]=d; out_ws[op]=w
        except Exception as e:
            print('skip',op,e)
    return out_df, out_ws

In [ ]:
# Batch run all 1D Hbb fits (can take time)
# HBB_ALL_DF, HBB_ALL_WS = run_all_hbb_1d(wc_min=-10, wc_max=10, n=401)
# summary = pd.DataFrame([{'op':op,'best_wc':d.loc[d.chi2_hbb.idxmin(),'wc'],'chi2_min':d['chi2_hbb'].min()} for op,d in HBB_ALL_DF.items()]).sort_values('chi2_min')
# summary.head(20)

## Plot every WC (1D) and every WC pair (2D)
These cells generate full plot sets for all operators in `ALL_OPS`.

**Warning:** full 2D scan over all pairs is computationally expensive.

In [ ]:
from itertools import combinations
OUTDIR = NBDIR/'outputs'/'hbb_repro'
OUTDIR.mkdir(parents=True, exist_ok=True)
print('output dir:', OUTDIR)

In [ ]:
# --- Run and plot ALL 1D fits ---
# Recommended grid for production: n=401 and wc range [-10,10] to match fit_plots display
HBB_ALL_DF, HBB_ALL_WS = run_all_hbb_1d(wc_min=-10, wc_max=10, n=401)
print('computed 1D fits:', len(HBB_ALL_DF))

for op, df in HBB_ALL_DF.items():
    fig, ax = plot_hbb_1d_auto(df)
    fig.savefig(OUTDIR/f'1D_{op}.png', dpi=150, bbox_inches='tight')
    plt.close(fig)

summary_1d = pd.DataFrame([
    {'op':op,
     'best_wc':float(df.loc[df.chi2_hbb.idxmin(),'wc']),
     'chi2_min':float(df['chi2_hbb'].min())}
    for op,df in HBB_ALL_DF.items()
]).sort_values('chi2_min')
summary_1d.to_csv(OUTDIR/'summary_1d.csv', index=False)
summary_1d.to_pickle(OUTDIR/'summary_1d.pkl')
print('saved all 1D plots + summary_1d')

In [ ]:
# --- Run and plot ALL 2D fits (all operator pairs) ---
# This can be very expensive: C(N,2) fits. For N=22 -> 231 scans.
# Reduce n for feasibility (e.g. n=61 or 81), then rerun selected pairs at higher granularity.
all_pairs = list(combinations(ALL_OPS, 2))
print('total pairs:', len(all_pairs))

HBB_ALL_2D = {}
failures = []
for op1, op2 in all_pairs:
    try:
        ws2d = hbb_stxs_fit_fullpath(coffea_fullpath, [op1, op2], wc_min=-10, wc_max=10, n=61)
        HBB_ALL_2D[(op1,op2)] = ws2d

        fig, ax = plot_hbb_2d_from_ws_auto(ws2d, op1, op2)
        fig.savefig(OUTDIR/f'2D_{op1}__{op2}.png', dpi=150, bbox_inches='tight')
        plt.close(fig)
    except Exception as e:
        failures.append((op1,op2,str(e)))

print('2D successful:', len(HBB_ALL_2D), 'failed:', len(failures))
if failures:
    fail_df = pd.DataFrame(failures, columns=['op1','op2','error'])
    fail_df.to_csv(OUTDIR/'failures_2d.csv', index=False)
    print('saved failures_2d.csv')

In [ ]:
# Build 2D summary table
rows=[]
for (op1,op2), ws2d in HBB_ALL_2D.items():
    best_pt, best_vals = min(ws2d.items(), key=lambda kv: kv[1]['total'][1])
    rows.append({
        'op1':op1,'op2':op2,
        'best_wc1':float(best_pt[0]),'best_wc2':float(best_pt[1]),
        'chi2_min':float(best_vals['total'][1])
    })
summary_2d = pd.DataFrame(rows).sort_values('chi2_min')
summary_2d.to_csv(OUTDIR/'summary_2d.csv', index=False)
summary_2d.to_pickle(OUTDIR/'summary_2d.pkl')
print('saved summary_2d')

## Automatic axis-range selection (robust 1σ/2σ framing)
The helpers below compute plotting windows directly from the Δχ² topology so plots automatically show the full 1σ and 2σ regions.

These auto-range plotting functions are the canonical plotting API in this notebook.

In [ ]:
def contiguous_intervals_1d(x, y, level):
    """Return contiguous x-intervals where y <= level (x sorted)."""
    mask = y <= level
    intervals=[]
    in_seg=False
    start=None
    for i,m in enumerate(mask):
        if m and not in_seg:
            in_seg=True; start=x[i]
        if in_seg and (i==len(mask)-1 or not mask[i+1]):
            end=x[i]
            intervals.append((float(start), float(end)))
            in_seg=False
    return intervals

def auto_xlim_from_deltachi2(x, y, pad_frac=0.10, min_pad=0.5):
    """Choose x-limits covering all points with Δχ² <= 5.99 (2σ for 2D-style threshold),
    fallback to <=1.0 if empty, then global range."""
    x=np.asarray(x, dtype=float)
    y=np.asarray(y, dtype=float)
    ymin=np.nanmin(y)
    dy=y-ymin

    sel = dy <= 5.99
    if not np.any(sel):
        sel = dy <= 1.0
    if np.any(sel):
        lo=float(np.nanmin(x[sel])); hi=float(np.nanmax(x[sel]))
    else:
        lo=float(np.nanmin(x)); hi=float(np.nanmax(x))

    span=max(hi-lo, 1e-9)
    pad=max(min_pad, pad_frac*span)
    return lo-pad, hi+pad

def auto_ylim_from_deltachi2(y, top_default=10.0):
    """Set y-range to clearly show 1σ and 2σ lines above minimum."""
    y=np.asarray(y, dtype=float)
    ymin=float(np.nanmin(y))
    # Need to include min+1 and min+5.99 with headroom
    target = ymin + 5.99
    ymax_data=float(np.nanmax(y))
    ymax=max(target*1.10, min(ymax_data, target*1.8), ymin+6.5)
    # Keep sane ceiling if scan huge
    ymax=min(ymax, max(top_default, ymin+20.0))
    return max(0.0, ymin-0.1), ymax

In [ ]:
def plot_hbb_1d_auto(df):
    x=df['wc'].to_numpy(dtype=float)
    y=df['chi2_hbb'].to_numpy(dtype=float)
    order=np.argsort(x); x=x[order]; y=y[order]
    y0=np.min(y); dy=y-y0

    fig, ax = plt.subplots(figsize=(6,4))
    ax.plot(x, y, label=df['op'].iloc[0], color='tab:blue')
    ax.axhline(y0+1.0, color='tab:red', ls='--', lw=1.2, label='1$\sigma$ (\Delta$\chi^2$=1)')
    ax.axhline(y0+4.0, color='tab:orange', ls=':', lw=1.2, label='2$\sigma$ (\Delta$\chi^2$=4)')

    xlo,xhi = auto_xlim_from_deltachi2(x, y)
    ylo,yhi = auto_ylim_from_deltachi2(y)
    ax.set_xlim(xlo,xhi)
    ax.set_ylim(ylo,yhi)

    # annotate intervals
    one_sig = contiguous_intervals_1d(x, dy, 1.0)
    two_sig = contiguous_intervals_1d(x, dy, 4.0)
    ax.text(0.02, 0.98, f'1$\sigma$: {one_sig}\n2$\sigma$: {two_sig}', transform=ax.transAxes, va='top', fontsize=8)

    ax.set_xlabel('Wilson coefficient value')
    ax.set_ylabel(r'$\chi^2$')
    ax.set_title(f"Hbb 1D $\chi^2$ scan: {df['op'].iloc[0]}")
    ax.legend(loc='best', fontsize=8)
    plt.tight_layout()
    return fig, ax

In [ ]:
def auto_xy_window_2d_from_deltachi2(ws_2d, level=5.99, pad_frac=0.10, min_pad=0.5):
    """Find compact x/y window covering all points with Δχ² <= level; robust fallback to full grid."""
    pts=[]
    vals=[]
    for (x,y),v in ws_2d.items():
        pts.append((float(x),float(y)))
        vals.append(float(v['total'][1]))
    pts=np.asarray(pts); vals=np.asarray(vals)
    dchi2 = vals - np.min(vals)
    sel = dchi2 <= level
    if not np.any(sel):
        sel = dchi2 <= 2.30
    if np.any(sel):
        xs=pts[sel,0]; ys=pts[sel,1]
    else:
        xs=pts[:,0]; ys=pts[:,1]

    def rng(a):
        lo=float(np.min(a)); hi=float(np.max(a)); span=max(hi-lo,1e-9); pad=max(min_pad,pad_frac*span); return lo-pad, hi+pad
    return rng(xs), rng(ys)

In [ ]:
def plot_hbb_2d_from_ws_auto(ws_2d, op1, op2, vmin=0, vmax=10):
    c1_vals = sorted(set(p[0] for p in ws_2d.keys()))
    c2_vals = sorted(set(p[1] for p in ws_2d.keys()))
    X, Y = np.meshgrid(c1_vals, c2_vals)
    Z = np.zeros((len(c2_vals), len(c1_vals)))
    for i,c2 in enumerate(c2_vals):
        for j,c1 in enumerate(c1_vals):
            Z[i,j] = ws_2d[(c1,c2)]['total'][1]
    Z = Z - np.min(Z)

    fig, ax = plt.subplots(figsize=(7,5))
    cmap = plt.cm.viridis.copy(); cmap.set_over('white')
    mesh = ax.pcolormesh(X, Y, Z, cmap=cmap, vmin=vmin, vmax=vmax)
    cbar = plt.colorbar(mesh, ax=ax, label=r'$\Delta\chi^2$')
    cbar.set_ticks([0,2.30,5.99,10])

    c1 = ax.contour(X, Y, Z, levels=[2.30], colors='red')
    c2 = ax.contour(X, Y, Z, levels=[5.99], colors='black', linestyles='dotted')
    ax.clabel(c1, inline=False, fontsize=9)
    ax.clabel(c2, inline=False, fontsize=9)

    best_pt = min(ws_2d.items(), key=lambda kv: kv[1]['total'][1])[0]
    ax.scatter([0],[0], color='red', marker='o', s=30, label='SM')
    ax.scatter([best_pt[0]],[best_pt[1]], color='red', marker='x', s=40, label='Best-fit')

    (xlo,xhi),(ylo,yhi)=auto_xy_window_2d_from_deltachi2(ws_2d, level=5.99)
    ax.set_xlim(xlo,xhi); ax.set_ylim(ylo,yhi)

    ax.set_xlabel(op1); ax.set_ylabel(op2)
    ax.set_title(f'Hbb 2D $\Delta\chi^2$: {op1} vs {op2}')
    ax.legend(loc='upper right', fontsize=8)
    plt.tight_layout()
    return fig, ax

In [ ]:
# Replace plot calls in batch loops to use auto-ranging versions.
# 1D example:
# fig, ax = plot_hbb_1d_auto(HBB_ALL_DF['cHW'])

# 2D example:
# ws2d = hbb_stxs_fit_fullpath(coffea_fullpath, ['cHW','cHWtil'], wc_min=-10, wc_max=10, n=121)
# fig, ax = plot_hbb_2d_from_ws_auto(ws2d, 'cHW', 'cHWtil')